# Rolling Returns Strategy for G10 Currency Futures

In this notebook, you will implement a **rolling returns strategy** using **G10 currency futures** data from major global markets. The aim is to construct a continuous return series by effectively managing futures contract rolls, ensuring seamless analysis and strategy development.

## Covered G10 Currency Futures and Exchanges

We have the data from the following **exchanges** and **contracts**:

- **Euro Futures (CME) 🇪🇺**
  - The **Euro FX Futures (6E)** are traded on the **Chicago Mercantile Exchange (CME)**.
  - These futures track the EUR/USD exchange rate, reflecting the relative strength of the Euro against the US Dollar.

- **Japanese Yen Futures (CME) 🇯🇵**
  - The **Japanese Yen Futures (6J)** are traded on the **Chicago Mercantile Exchange (CME)**.
  - These futures track the JPY/USD exchange rate, influenced by Japan's monetary policy and trade flows.

- **British Pound Futures (CME) 🇬🇧**
  - The **British Pound Futures (6B)** are traded on the **Chicago Mercantile Exchange (CME)**.
  - These futures track the GBP/USD exchange rate, reflecting UK economic conditions and monetary policy.

- **Australian Dollar Futures (CME) 🇦🇺**
  - The **Australian Dollar Futures (6A)** are traded on the **Chicago Mercantile Exchange (CME)**.
  - These futures track the AUD/USD exchange rate, influenced by commodity prices and Asian economic conditions.

- **Canadian Dollar Futures (CME) 🇨🇦**
  - The **Canadian Dollar Futures (6C)** are traded on the **Chicago Mercantile Exchange (CME)**.
  - These futures track the CAD/USD exchange rate, affected by oil prices and US-Canada trade relations.

- **Swiss Franc Futures (CME) 🇨🇭**
  - The **Swiss Franc Futures (6S)** are traded on the **Chicago Mercantile Exchange (CME)**.
  - These futures track the CHF/USD exchange rate, influenced by Swiss monetary policy and safe-haven flows.

- **New Zealand Dollar Futures (CME) 🇳🇿**
  - The **New Zealand Dollar Futures (6N)** are traded on the **Chicago Mercantile Exchange (CME)**.
  - These futures track the NZD/USD exchange rate, affected by dairy prices and Asian economic conditions.

- **Norwegian Krone Futures (CME) 🇳🇴**
  - The **Norwegian Krone Futures (6N)** are traded on the **Chicago Mercantile Exchange (CME)**.
  - These futures track the NOK/USD exchange rate, influenced by oil prices and Norwegian economic conditions.

- **Swedish Krona Futures (CME) 🇸🇪**
  - The **Swedish Krona Futures (6S)** are traded on the **Chicago Mercantile Exchange (CME)**.
  - These futures track the SEK/USD exchange rate, affected by Swedish monetary policy and European economic conditions.

## Strategy Overview

The strategy will:
- **Roll contracts efficiently** by monitoring **open interest and volume** to determine optimal roll-over dates.
- **Construct a continuous time series** of currency futures **returns** to facilitate backtesting and quantitative analysis.
- **Analyze correlation structures** between currency pairs to assess market relationships and potential diversification benefits.


In [1]:
# Import the required libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
import seaborn as sns

In [2]:
# Read the CSV files for the price, volume and open interest

df_price = pd.read_csv("g10_futures_prices.csv")
df_volume = pd.read_csv("g10_futures_volume.csv")
df_open_interest = pd.read_csv("g10_futures_open_interest.csv")

In [3]:
df_price.head()

,Unnamed: 0,BPH08 Curncy,JYZ06 Curncy,JYH07 Curncy,JYM07 Curncy,JYU07 Curncy,JYZ07 Curncy,JYH08 Curncy,BPZ06 Curncy,BPH07 Curncy,...,ADU04 Curncy,SFU04 Curncy,NOU04 Curncy,SEU04 Curncy,NVU25 Curncy,NOU25 Curncy,SEU25 Curncy,NVZ25 Curncy,NOZ25 Curncy,SEZ25 Curncy
0,2000-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


As you can see above the date column has a weird name for the column title, lets change that below.

In [4]:
# Rename the Date column
df_price = df_price.rename(columns = {'Unnamed: 0': 'Date'})
df_volume = df_volume.rename(columns = {'Unnamed: 0': 'Date'})
df_open_interest = df_open_interest.rename(columns = {'Unnamed: 0': 'Date'})

The dates have the format mm/dd/yyyy so lets make sure Pandas knows what the dates mean

In [5]:
# Set to date time
df_price['Date'] = pd.to_datetime(df_price['Date'], format='%Y-%m-%d')
df_volume['Date'] = pd.to_datetime(df_volume['Date'], format='%Y-%m-%d')
df_open_interest['Date'] = pd.to_datetime(df_open_interest['Date'], format='%Y-%m-%d')

In [6]:
# Set to date time
df_price['Date'] = pd.to_datetime(df_price['Date'], format='%Y/%m/%d')
df_volume['Date'] = pd.to_datetime(df_volume['Date'], format='%Y/%m/%d')
df_open_interest['Date'] = pd.to_datetime(df_open_interest['Date'], format='%Y/%m/%d')

We would also like to have the date column as our index so lets do that here

In [7]:
df_volume.set_index("Date", inplace = True)
df_price.set_index("Date", inplace = True)
df_open_interest.set_index("Date", inplace = True)

Now lets look at how the dataframe looks like 

In [8]:
df_price.head()

,BPH08 Curncy,JYZ06 Curncy,JYH07 Curncy,JYM07 Curncy,JYU07 Curncy,JYZ07 Curncy,JYH08 Curncy,BPZ06 Curncy,BPH07 Curncy,BPM07 Curncy,...,ADU04 Curncy,SFU04 Curncy,NOU04 Curncy,SEU04 Curncy,NVU25 Curncy,NOU25 Curncy,SEU25 Curncy,NVZ25 Curncy,NOZ25 Curncy,SEZ25 Curncy
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Currency Futures Contracts and Expiry Codes

Currency futures contracts typically expire every **three** months, following a quarterly cycle with expiry months in **March**, **June**, **September**, and **December**.

The columns in our dataset may appear coded, but they are actually labeled systematically. Let’s break down the code **ECH10** as an example.

- The first two letters (**EC**) represent the **Euro** currency code (this is specific to Clearport).
- The letter **H** represents the **expiry month** of **March**.
- The number **10** denotes the **year 2000**.

This structure allows for quick identification of the contract’s underlying currency, expiry date, and trading period.

### Currency Futures and Their Codes (2000–2025)

Our dataset includes **currency futures contracts** for major global currencies, spanning from **2000 to 2025**. The contract codes follow this format:

| Country/Currency        | Currency Code (Clearport)  | Example Code (March 2020) | 
|-------------------------|----------------------------|---------------------------|
| **Euro**                | EC                         | **ECH20**                  |
| **Pound Sterling**      | BP                         | **BPH20**                  |
| **Japanese Yen**        | JY                         | **JYH20**                  |
| **Australian Dollar**   | AD                         | **ADH20**                  |
| **New Zealand Dollar**  | NV                         | **NVH20**                  |
| **Norwegian Kroner**    | NO                         | **NOH20**                  |
| **Swedish Kroner**      | SE                         | **SEH20**                  |
| **Canadian Dollar**     | CD                         | **CDH20**                  |
| **Swiss Franc**         | SF                         | **SFH20**                  |

This pattern is repeated **every quarter** from **2000 to 2025**, adjusting the expiry year accordingly (e.g., **ECM23** for June 2023).

### Expiry Month Codes

Each futures contract follows the standard **quarterly expiration cycle**:

- **H** for **March**
- **M** for **June**
- **U** for **September**
- **Z** for **December**

### Rolling and Analysis

Currency futures contracts **expire quarterly**, so maintaining a continuous return series requires **rolling** into the next liquid contract before expiry. This dataset allows us to analyze **historical currency futures performance**, track **foreign exchange trends**, and develop **macro trading strategies**.

By systematically managing these futures, we can construct a **continuous currency return series** for **quantitative research and risk analysis** from **2000 to 2025**.


In [9]:
df_price

,BPH08 Curncy,JYZ06 Curncy,JYH07 Curncy,JYM07 Curncy,JYU07 Curncy,JYZ07 Curncy,JYH08 Curncy,BPZ06 Curncy,BPH07 Curncy,BPM07 Curncy,...,ADU04 Curncy,SFU04 Curncy,NOU04 Curncy,SEU04 Curncy,NVU25 Curncy,NOU25 Curncy,SEU25 Curncy,NVZ25 Curncy,NOZ25 Curncy,SEZ25 Curncy
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,57.515,9.4075,10.0125,57.630,9.4000,10.0550
2025-03-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,57.270,9.3525,9.8975,57.370,9.3450,9.9375
2025-03-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,57.715,9.4000,9.9650,57.825,9.3925,10.0075


# Big Hint! Make sure you stitch returns not prices!